# 02 - Changepoint Detection

We compare five changepoint detection methods on **real STOXX 600 stocks**:

| # | Method | Idea |
|---|--------|------|
| 1 | **Brownian Motion (CUSUM)** | Page's sequential CUSUM — accumulates standardised deviations from a rolling reference mean. |
| 2 | **Jump Process** | Returns modelled as diffusion + jumps; flag days where the rolling z-score exceeds a threshold. |
| 3 | **Econometrics (rolling t-test)** | Welch two-sample t-test between pre/post windows. |
| 4 | **MA Crossover** | Signal when short MA crosses long MA — proxy for trend reversal. |
| 5 | **GP Matérn 3/2 + Changepoint Kernel** | Paper method (Wood, Roberts & Zohren 2022). |

> **Key change vs raw returns**: CPD is run on **relative (idiosyncratic) returns** `1d_rel_ret = r_stock - r_benchmark_ew`  
> to focus on stock-specific regime changes, not market-wide shocks.

---
## 0. Imports

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from scipy import stats
from typing import Tuple, List, Dict

plt.style.use("seaborn-whitegrid")   # compatible toutes versions matplotlib
plt.rcParams.update({"figure.figsize": (14, 4), "figure.dpi": 100})

---
## 1. Load data & define known events

In [ ]:
# Charger price + 1d_rel_ret (returns idiosyncratiques)
df = pd.read_csv(
    "../data/processed/stoxx600/stoxx600_processed.csv",
    parse_dates=["date"],
    usecols=["date", "ticker", "price", "1d_rel_ret"],
)

STOCKS = [
    "ASML NA",   # Tech — Netherlands
    "TTE FP",    # Energy — France
    "BARC LN",   # Banks — UK
    "SAP GY",    # Tech — Germany
    "NOVN SE",   # Pharma — Switzerland
    "SAN SQ",    # Banks — Spain
    "ENEL IM",   # Utilities — Italy
    "NOVOB DC",  # Pharma — Denmark
]

# Known European equity regime changes (macro shocks)
# Note: these are SYSTEMIC events — idiosyncratic shocks cannot be scored this way
KNOWN_EVENTS = {
    "SNB removes EUR/CHF floor":                pd.Timestamp("2015-01-15"),
    "ECB QE launch":                            pd.Timestamp("2015-01-22"),
    "Greek capital controls / bank holiday":    pd.Timestamp("2015-06-29"),
    "China Black Monday":                       pd.Timestamp("2015-08-24"),
    "Brexit vote":                              pd.Timestamp("2016-06-24"),
    "Trump election 2016":                      pd.Timestamp("2016-11-09"),
    "Volmageddon (XIV implosion)":              pd.Timestamp("2018-02-05"),
    "Italy BTP selloff / populist coalition":   pd.Timestamp("2018-05-29"),
    "Q4 2018 selloff":                          pd.Timestamp("2018-10-10"),
    "COVID crash":                              pd.Timestamp("2020-02-24"),
    "COVID recovery":                           pd.Timestamp("2020-03-23"),
    "Rate-hike selloff":                        pd.Timestamp("2022-01-05"),
    "Russia-Ukraine war":                       pd.Timestamp("2022-02-24"),
    "UK mini-budget (Truss)":                   pd.Timestamp("2022-09-26"),
    "SVB collapse":                             pd.Timestamp("2023-03-10"),
    "Credit Suisse / UBS rescue":               pd.Timestamp("2023-03-15"),
    "Israel-Hamas war":                         pd.Timestamp("2023-10-09"),
    "French snap election (Macron dissolution)": pd.Timestamp("2024-06-10"),
    "Yen carry trade unwind":                   pd.Timestamp("2024-08-05"),
    "Trump election 2024":                      pd.Timestamp("2024-11-06"),
    "DeepSeek AI shock":                        pd.Timestamp("2025-01-27"),
    "German debt brake / Zeitenwende":          pd.Timestamp("2025-03-05"),
    "Trump Liberation Day tariffs":             pd.Timestamp("2025-04-03"),
    "Liberation Day selloff nadir":             pd.Timestamp("2025-04-07"),
    "Tariff 90-day pause rebound":              pd.Timestamp("2025-04-09"),
}

print(f"Loaded {df['ticker'].nunique()} tickers, using {len(STOCKS)} for analysis")
print(f"Known events: {len(KNOWN_EVENTS)}")

---
## 2. load_stock — utilise les returns relatifs

In [ ]:
def load_stock(ticker: str) -> Tuple[pd.Series, np.ndarray, np.ndarray, List[int]]:
    """
    Charge un stock et retourne ses returns RELATIFS (idiosyncratiques).
    returns = 1d_rel_ret = r_stock - r_benchmark_ew
    """
    s = (
        df.loc[df["ticker"] == ticker, ["date", "price", "1d_rel_ret"]]
        .sort_values("date")
        .reset_index(drop=True)
    )
    dates   = s["date"]
    prices  = s["price"].values
    # Returns relatifs — dropna pour les premiers jours
    rel_ret = s["1d_rel_ret"].fillna(0.0).values

    # Mapper les evenements connus -> indices dans le tableau de returns
    event_idxs = []
    for ev_date in KNOWN_EVENTS.values():
        diffs   = (dates - ev_date).abs()
        nearest = diffs.idxmin()
        if diffs[nearest].days <= 7:
            idx = min(nearest, len(rel_ret) - 1)
            event_idxs.append(idx)

    return dates, prices, rel_ret, sorted(set(event_idxs))


dates_ex, prices_ex, returns_ex, events_ex = load_stock("ASML NA")
print(f"ASML NA: {len(returns_ex)} returns, {len(events_ex)} events matched")
print(f"Return type: idiosyncratique (relatif benchmark EW)")

In [ ]:
# Visualise stock principal
PRIMARY = "ASML NA"
dates_p, prices_p, returns_p, events_p = load_stock(PRIMARY)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(dates_p, prices_p, linewidth=0.8)
axes[0].set_title(f"{PRIMARY} — price")
axes[0].set_ylabel("Price")

axes[1].plot(dates_p, returns_p, linewidth=0.4, alpha=0.6, color="C1")
axes[1].set_title(f"{PRIMARY} — relative daily returns (idiosyncratic)")
axes[1].set_ylabel("Rel. Return")

for ev_name, ev_date in KNOWN_EVENTS.items():
    for ax in axes:
        ax.axvline(ev_date, color="red", ls="--", lw=0.8, alpha=0.7)
axes[0].plot([], [], color="red", ls="--", lw=0.8, label="Known event")
axes[0].legend()
plt.tight_layout()
plt.show()

---
## 3. Evaluation framework

**Une seule définition** — version complète avec `n_tp`, `n_fp`, `mean_delay`.

In [ ]:
def evaluate_detections(
    detected: List[int],
    true_cps: List[int],
    tol: int = 20,
) -> Dict:
    """Score detected changepoints against ground truth."""
    if len(detected) == 0:
        return {
            "n_det"      : 0,
            "n_tp"       : 0,
            "n_fp"       : 0,
            "fpr"        : 0.0,
            "recall"     : 0.0,
            "delays"     : [],
            "mean_delay" : float("nan"),
        }

    tp          = set()
    matched_det = set()
    delays      = []

    for i, ev in enumerate(true_cps):
        for d in detected:
            if abs(d - ev) <= tol and d not in matched_det:
                tp.add(i)
                matched_det.add(d)
                delays.append(abs(d - ev))   # abs() -> delai toujours positif
                break

    n_det      = len(detected)
    n_tp       = len(matched_det)
    n_fp       = n_det - n_tp
    fpr        = n_fp / n_det             if n_det > 0       else 0.0
    recall     = len(tp) / len(true_cps)  if len(true_cps) > 0 else 0.0
    mean_delay = np.mean(delays)          if delays          else float("nan")

    return {
        "n_det"      : n_det,
        "n_tp"       : n_tp,
        "n_fp"       : n_fp,
        "fpr"        : fpr,
        "recall"     : recall,
        "delays"     : delays,
        "mean_delay" : mean_delay,
    }

---
## 4. Detectors

In [ ]:
# ── Method 1: CUSUM ───────────────────────────────────────────────────────────
def detect_cusum(
    returns: np.ndarray,
    ref_window: int = 120,
    threshold: float = 3.0,
    cooldown: int = 30,
) -> List[int]:
    """Page's two-sided CUSUM. Params optimises par grid search."""
    n = len(returns)
    detections = []
    last_det   = -cooldown - 1
    k          = 0.5
    s_pos, s_neg = 0.0, 0.0
    for t in range(ref_window, n):
        ref = returns[t - ref_window : t]
        mu, sigma = ref.mean(), ref.std(ddof=1)
        if sigma < 1e-12:
            continue
        z     = (returns[t] - mu) / sigma
        s_pos = max(0.0, s_pos + z - k)
        s_neg = max(0.0, s_neg - z - k)
        if max(s_pos, s_neg) > threshold and (t - last_det) > cooldown:
            detections.append(t)
            last_det   = t
            s_pos = s_neg = 0.0
    return detections


# ── Method 2: Jump ────────────────────────────────────────────────────────────
def detect_jump(
    returns: np.ndarray,
    window: int = 120,
    threshold: float = 2.5,
    cooldown: int = 20,
) -> List[int]:
    """Jump detection via rolling z-score. Params optimises par grid search."""
    n = len(returns)
    detections = []
    last_det   = -cooldown - 1
    for t in range(window, n):
        w = returns[t - window : t]
        mu, sigma = w.mean(), w.std(ddof=1)
        if sigma < 1e-12:
            continue
        if abs(returns[t] - mu) / sigma > threshold and (t - last_det) > cooldown:
            detections.append(t)
            last_det = t
    return detections


# ── Method 3: t-test ──────────────────────────────────────────────────────────
def detect_ttest(
    returns: np.ndarray,
    window: int = 30,
    alpha: float = 0.001,
    cooldown: int = 20,
) -> List[int]:
    """Rolling Welch t-test for mean shift."""
    n = len(returns)
    detections = []
    last_det   = -cooldown - 1
    for t in range(2 * window, n):
        pre  = returns[t - 2 * window : t - window]
        post = returns[t - window : t]
        _, p = stats.ttest_ind(pre, post, equal_var=False)
        if p < alpha and (t - last_det) > cooldown:
            detections.append(t)
            last_det = t
    return detections


# ── Method 4: MA Crossover ────────────────────────────────────────────────────
def detect_ma_crossover(
    returns: np.ndarray,
    fast: int = 20,
    slow: int = 60,
    cooldown: int = 30,
) -> List[int]:
    """Detecte un croisement MA rapide / MA lente. Params optimises par grid search."""
    n  = len(returns)
    detections = []
    last_det   = -cooldown - 1
    ma_fast = np.full(n, np.nan)
    ma_slow = np.full(n, np.nan)
    for t in range(slow, n):
        ma_fast[t] = returns[t - fast : t].mean()
        ma_slow[t] = returns[t - slow : t].mean()
    for t in range(slow + 1, n):
        if np.isnan(ma_fast[t]) or np.isnan(ma_slow[t]):
            continue
        cross_up   = ma_fast[t] > ma_slow[t] and ma_fast[t-1] <= ma_slow[t-1]
        cross_down = ma_fast[t] < ma_slow[t] and ma_fast[t-1] >= ma_slow[t-1]
        if (cross_up or cross_down) and (t - last_det) > cooldown:
            detections.append(t)
            last_det = t
    return detections


# ── Method 5: GP CPD (paper) ──────────────────────────────────────────────────
from src.cpd import cpd_scores

def detect_gp_cpd(
    returns: np.ndarray,
    lbw: int = 21,
    nu_threshold: float = 0.85,
    gamma_min: float = 0.5,
    cooldown: int = 20,
    stride: int = 1,
) -> Tuple[List[int], np.ndarray, np.ndarray]:
    """GP-based CPD — Wood, Roberts & Zohren (2022)."""
    n         = len(returns)
    nu_arr    = np.full(n, np.nan)
    gamma_arr = np.full(n, np.nan)
    detections = []
    last_det   = -cooldown - 1
    for t in range(lbw, n, stride):
        window        = returns[t - lbw : t]
        nu, gamma     = cpd_scores(window, lbw)
        nu_arr[t]     = nu
        gamma_arr[t]  = gamma
        if nu > nu_threshold and gamma > gamma_min and (t - last_det) > cooldown:
            detections.append(t)
            last_det = t
    return detections, nu_arr, gamma_arr

print("Tous les detecteurs definis.")

---
## 5. Run detectors on primary stock

In [ ]:
# CUSUM
det_cusum = detect_cusum(returns_p, ref_window=120, threshold=3.0, cooldown=30)
res_cusum = evaluate_detections(det_cusum, events_p, tol=20)
print(f"CUSUM  : {res_cusum['n_det']} det | FPR={res_cusum['fpr']:.0%} | Recall={res_cusum['recall']:.0%} | Delay={res_cusum['mean_delay']:.1f}d")

# Jump
det_jump = detect_jump(returns_p, window=120, threshold=2.5, cooldown=20)
res_jump = evaluate_detections(det_jump, events_p, tol=20)
print(f"Jump   : {res_jump['n_det']} det | FPR={res_jump['fpr']:.0%} | Recall={res_jump['recall']:.0%} | Delay={res_jump['mean_delay']:.1f}d")

# t-test
det_ttest = detect_ttest(returns_p, window=20, alpha=0.01, cooldown=30)
res_ttest = evaluate_detections(det_ttest, events_p, tol=20)
print(f"t-test : {res_ttest['n_det']} det | FPR={res_ttest['fpr']:.0%} | Recall={res_ttest['recall']:.0%} | Delay={res_ttest['mean_delay']:.1f}d")

# MA Crossover
det_ma = detect_ma_crossover(returns_p, fast=20, slow=60, cooldown=30)
res_ma = evaluate_detections(det_ma, events_p, tol=20)
print(f"MA     : {res_ma['n_det']} det | FPR={res_ma['fpr']:.0%} | Recall={res_ma['recall']:.0%} | Delay={res_ma['mean_delay']:.1f}d")

# GP CPD
det_gp, nu_arr, gamma_arr = detect_gp_cpd(
    returns_p, lbw=21, nu_threshold=0.85, gamma_min=0.5, cooldown=30, stride=10
)
res_gp = evaluate_detections(det_gp, events_p, tol=20)
print(f"GP CPD : {res_gp['n_det']} det | FPR={res_gp['fpr']:.0%} | Recall={res_gp['recall']:.0%} | Delay={res_gp['mean_delay']:.1f}d")

In [ ]:
# Tableau comparatif single-stock
results_single = {
    "CUSUM"       : res_cusum,
    "Jump"        : res_jump,
    "t-test"      : res_ttest,
    "MA Crossover": res_ma,
    "GP Matérn"   : res_gp,
}

summary = pd.DataFrame({
    name: {
        "Detections"     : r["n_det"],
        "True Positives" : r["n_tp"],
        "False Positives": r["n_fp"],
        "FPR"            : f"{r['fpr']:.1%}",
        "Recall"         : f"{r['recall']:.1%}",
        "Mean delay (d)" : f"{r['mean_delay']:.1f}" if not np.isnan(r['mean_delay']) else "—",
    }
    for name, r in results_single.items()
}).T

print(f"Results on {PRIMARY} ({len(events_p)} known events, tol=±20d)")
summary

In [ ]:
# Toutes les methodes sur un graphique
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_p, prices_p, lw=0.8, color="black", alpha=0.5, label="Price")

for ev_date in KNOWN_EVENTS.values():
    ax.axvline(ev_date, color="red", ls="--", lw=1, alpha=0.7)
ax.plot([], [], color="red", ls="--", lw=1, label="Known event")

all_dets = [
    (det_cusum, "CUSUM",        "C0"),
    (det_jump,  "Jump",         "C2"),
    (det_ttest, "t-test",       "C1"),
    (det_ma,    "MA Crossover", "C3"),
    (det_gp,    "GP CPD",       "C4"),
]
for dets, name, color in all_dets:
    for d in dets:
        ax.axvline(dates_p.iloc[d], color=color, ls=":", lw=0.6, alpha=0.5)
    if dets:
        ax.plot([], [], color=color, ls=":", lw=2, label=f"{name} ({len(dets)})")

ax.set_title(f"{PRIMARY} — all CPD methods vs known events (idiosyncratic returns)")
ax.set_ylabel("Price")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# GP : prix + score nu + score gamma
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(dates_p, prices_p, lw=0.7, color="black", alpha=0.5)
for ev_date in KNOWN_EVENTS.values():
    axes[0].axvline(ev_date, color="red", ls="--", lw=0.7, alpha=0.6)
for d in det_gp:
    axes[0].axvline(dates_p.iloc[d], color="C4", ls=":", lw=0.7)
axes[0].plot([], [], color="red",  ls="--", lw=1, label="Known event")
axes[0].plot([], [], color="C4",   ls=":",  lw=1, label=f"GP CPD ({len(det_gp)})")
axes[0].set_title(f"GP Matérn 3/2 CPD — {PRIMARY} (idiosyncratic returns)")
axes[0].set_ylabel("Price")
axes[0].legend()

valid = ~np.isnan(nu_arr)
t_idx = np.where(valid)[0]
axes[1].plot(dates_p.iloc[t_idx], nu_arr[t_idx], lw=0.7, color="C3")
axes[1].axhline(0.85, color="gray", ls="--", lw=0.8, label="threshold ν=0.85")
for ev_date in KNOWN_EVENTS.values():
    axes[1].axvline(ev_date, color="red", ls="--", lw=0.5, alpha=0.4)
axes[1].set_ylabel("Severity (ν)")
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend()

axes[2].plot(dates_p.iloc[t_idx], gamma_arr[t_idx], lw=0.7, color="C4")
axes[2].axhline(0.5, color="gray", ls="--", lw=0.8, label="gamma_min=0.5")
for ev_date in KNOWN_EVENTS.values():
    axes[2].axvline(ev_date, color="red", ls="--", lw=0.5, alpha=0.4)
axes[2].set_ylabel("Location (γ)")
axes[2].set_ylim(-0.05, 1.05)
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 6. Multi-stock evaluation

In [ ]:
%%time
METHOD_NAMES = ["CUSUM", "Jump", "t-test", "MA Crossover", "GP Matérn 3/2"]
multi = {m: {"fpr": [], "recall": [], "delay": []} for m in METHOD_NAMES}

for ticker in STOCKS:
    _, _, ret, ev_idx = load_stock(ticker)
    if len(ev_idx) == 0:
        continue

    detectors = {
        "CUSUM"        : detect_cusum(ret, ref_window=120, threshold=3.0, cooldown=30),
        "Jump"         : detect_jump(ret, window=120, threshold=2.5, cooldown=20),
        "t-test"       : detect_ttest(ret, window=20, alpha=0.01, cooldown=30),
        "MA Crossover" : detect_ma_crossover(ret, fast=20, slow=60, cooldown=30),
        "GP Matérn 3/2": detect_gp_cpd(ret, lbw=21, nu_threshold=0.85, cooldown=30, stride=10)[0],
    }

    for name, dets in detectors.items():
        ev = evaluate_detections(dets, ev_idx, tol=20)
        multi[name]["fpr"].append(ev["fpr"])
        multi[name]["recall"].append(ev["recall"])
        multi[name]["delay"].append(ev["mean_delay"])

    print(f"  {ticker}: done")

print(f"\nDone ({len(STOCKS)} stocks).")

In [ ]:
# Tableau agrege
multi_summary = pd.DataFrame({
    name: {
        "Mean FPR"      : f"{np.mean(v['fpr']):.1%}",
        "Std FPR"       : f"{np.std(v['fpr']):.1%}",
        "Mean Recall"   : f"{np.mean(v['recall']):.1%}",
        "Std Recall"    : f"{np.std(v['recall']):.1%}",
        "Mean delay (d)": f"{np.nanmean(v['delay']):.1f}",
    }
    for name, v in multi.items()
}).T

print(f"Aggregated across {len(STOCKS)} STOXX 600 stocks — idiosyncratic returns")
multi_summary

In [ ]:
# Bar chart FPR / Recall / Delay
methods = list(multi.keys())
fprs    = [np.mean(multi[m]["fpr"])        for m in methods]
recalls = [np.mean(multi[m]["recall"])     for m in methods]
delays  = [np.nanmean(multi[m]["delay"])   for m in methods]
colors  = ["C0", "C2", "C1", "C3", "C4"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(methods, fprs, color=colors)
axes[0].set_title("False Positive Rate (↓ better)")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(methods, rotation=20, ha="right")

axes[1].bar(methods, recalls, color=colors)
axes[1].set_title("Recall (↑ better)")
axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(methods, rotation=20, ha="right")

axes[2].bar(methods, delays, color=colors)
axes[2].set_title("Mean Delay in days (↓ better)")
axes[2].set_xticklabels(methods, rotation=20, ha="right")

plt.suptitle(f"CPD Method Comparison — {len(STOCKS)} STOXX 600 stocks (idiosyncratic returns)")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter FPR vs Recall
fig, ax = plt.subplots(figsize=(7, 5))
for (name, v), c in zip(multi.items(), colors):
    ax.scatter(
        np.mean(v["fpr"]), np.mean(v["recall"]),
        s=160, color=c, zorder=5, edgecolors="black", linewidth=0.5,
    )
    ax.annotate(name, (np.mean(v["fpr"]), np.mean(v["recall"])),
                textcoords="offset points", xytext=(8, 6), fontsize=9)

ax.set_xlabel("Mean FPR (lower is better)")
ax.set_ylabel("Mean Recall (higher is better)")
ax.set_title(f"FPR vs Recall — {len(STOCKS)} real STOXX 600 stocks")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
# Ideal point
ax.scatter([0], [1], marker="*", s=300, color="gold", zorder=6, label="Ideal")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Conclusion

**Pourquoi on utilise le GP dans le pipeline DMN ?**

In [ ]:
print("=" * 60)
print("CONCLUSION — Choix du detecteur pour le pipeline DMN")
print("=" * 60)
print()
print("1. SCORE CONTINU")
print("   Le GP sort nu in (0,1) et gamma in (0,1).")
print("   Le LSTM apprend par gradient -> score continu indispensable.")
print("   Les methodes naives sortent un flag binaire 0/1.")
print()
print("2. CALIBRATION BAYESIENNE")
print("   nu = comparaison de log-vraisemblance marginale")
print("   (kernel stationnaire vs kernel changepoint)")
print("   Approxime un facteur de Bayes -> score interpetable.")
print()
print("3. MATERN 3/2 CAPTURE 3 TYPES DE RUPTURES")
print("   - Changement de correlation (input scale lambda)")
print("   - Changement de mean-reversion (output scale sigma_h)")
print("   - Changement de volatilite (noise sigma_n)")
print()
print("4. RETURNS IDIOSYNCRATIQUES")
print("   CPD tourne sur 1d_rel_ret = r_stock - r_benchmark_ew")
print("   -> detecte les ruptures propres au stock, pas les chocs macro.")
print()
print("Next step: brancher nu et gamma comme inputs du LSTM (notebook 03).")